# Hardware Measurement (portable) — run this SAME notebook on each free runtime
Targets: **Colab T4**, **Kaggle P100** (Settings → Accelerator P100, Internet ON), and one **CPU-only** runtime (Colab CPU or GitHub Codespaces).

Measures the cheap (EfficientNet-B0) vs heavy (ConvNeXt-Tiny) cost four ways — GMACs (constant), GPU batched latency, GPU energy (NVML via CodeCarbon), CPU single-image latency — auto-labels the row with the detected device, and appends to `hw_results.csv`. Merge the CSVs from the runs into the paper's hardware table.

Notes:
- **No training needed** — latency/energy don't depend on weights (random init used).
- On CPU-only runtimes the GPU rows are skipped automatically; CPU *energy* is intentionally not claimed (RAPL is unreadable in most containers — report CPU latency only, say so in the paper).
- Runtime: ~5–8 min on GPU, ~5–10 min on a 2-core CPU.

In [ ]:
!pip -q install timm fvcore codecarbon
import os, time, platform, csv
import numpy as np, torch, timm
from fvcore.nn import FlopCountAnalysis

has_gpu = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_gpu else "none"
try:
    import multiprocessing; n_cores = multiprocessing.cpu_count()
except Exception: n_cores = -1
cpu_model = "unknown"
try:
    with open("/proc/cpuinfo") as f:
        for line in f:
            if "model name" in line:
                cpu_model = line.split(":",1)[1].strip(); break
except Exception: pass
n_threads = torch.get_num_threads()
def detect_env():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"):
        return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "other"
env = detect_env()
print(f"env={env}  gpu={gpu_name}  torch={torch.__version__}")
print(f"cpu_model={cpu_model}  cores={n_cores}  torch_threads={n_threads}")

cheap = timm.create_model("efficientnet_b0", pretrained=False, num_classes=81).eval()
heavy = timm.create_model("convnext_tiny",   pretrained=False, num_classes=81).eval()
d1 = torch.randn(1,3,224,224)
G_CHEAP = FlopCountAnalysis(cheap,d1).total()/1e9
G_HEAVY = FlopCountAnalysis(heavy,d1).total()/1e9
print(f"GMACs cheap={G_CHEAP:.3f} heavy={G_HEAVY:.3f} ratio={G_HEAVY/G_CHEAP:.1f}x")

In [ ]:
def gpu_latency(model, bs=64, warmup=10, iters=50):
    m=model.to("cuda"); x=torch.randn(bs,3,224,224,device="cuda")
    with torch.no_grad():
        for _ in range(warmup): m(x)
        torch.cuda.synchronize(); t0=time.time()
        for _ in range(iters): m(x)
        torch.cuda.synchronize()
    return (time.time()-t0)/(iters*bs)*1000

def gpu_energy(model, bs=64, iters=200):
    from codecarbon import EmissionsTracker
    m=model.to("cuda"); x=torch.randn(bs,3,224,224,device="cuda")
    with torch.no_grad():
        for _ in range(10): m(x)
        torch.cuda.synchronize()
    tr=EmissionsTracker(measure_power_secs=1,save_to_file=False,log_level="error"); tr.start()
    with torch.no_grad():
        for _ in range(iters): m(x)
        torch.cuda.synchronize()
    return tr.stop()/(iters*bs)

torch.set_num_threads(os.cpu_count())   # pin threads = cores (recorded in device label)
def cpu_latency(model, warmup=2, iters=None):
    m=model.to("cpu").eval(); x=torch.randn(1,3,224,224)
    with torch.no_grad():
        t=time.time(); m(x); per=time.time()-t                     # calibrate
        it = iters or max(5, min(15, int(30/max(per,0.05))))       # target <=30s per model
        for _ in range(warmup): m(x)
        t0=time.time()
        for _ in range(it): m(x)
    return (time.time()-t0)/it*1000

rows=[("GMACs", G_CHEAP, G_HEAVY)]
if has_gpu:
    lc,lh = gpu_latency(cheap), gpu_latency(heavy)
    rows.append(("gpu_latency_ms", lc, lh))
    try:
        ec,eh = gpu_energy(cheap), gpu_energy(heavy)
        rows.append(("gpu_energy_kgco2e", ec, eh))
    except Exception as ex:
        print("gpu energy skipped:", ex)
    cheap.to("cpu"); heavy.to("cpu"); torch.cuda.empty_cache()
cc,ch = cpu_latency(cheap), cpu_latency(heavy)
rows.append(("cpu_latency_ms", cc, ch))
for name,a,b in rows:
    print(f"{name:20s} cheap={a:.4g}  heavy={b:.4g}  ratio(E_cheap/E_heavy)={a/b:.3f}")

In [ ]:
# Savings check at routing fractions + append labeled CSV row(s)
FRACS=(0.30, 0.40, 0.43, 0.60)
print(f"{'metric':20s} | ratio | " + "  ".join(f"f={f}" for f in FRACS))
out=[]
for name,a,b in rows:
    r=a/b
    sav=[100*(1-(r+f)) for f in FRACS]
    print(f"{name:20s} | {r:5.3f} | " + "  ".join(f"{s:+5.0f}%" for s in sav))
    out.append(dict(env=env, device=(gpu_name if name.startswith('gpu') else f"{cpu_model} ({n_cores}c/{torch.get_num_threads()}t)"),
                    metric=name, cheap=a, heavy=b, ratio=r,
                    **{f"save_f{f}": s for f,s in zip(FRACS,sav)}))

fn="hw_results.csv"; new=not os.path.exists(fn)
with open(fn,"a",newline="") as f:
    w=csv.DictWriter(f, fieldnames=list(out[0].keys()))
    if new: w.writeheader()
    for r_ in out: w.writerow(r_)
print(f"\\nappended {len(out)} rows -> {fn}  (download this file; merge across runtimes)")
print("cascade saves iff ratio < 1 - f; positive % = cheaper than always-heavy")

### Collection plan
1. Colab **T4** → run all → download `hw_results.csv`
2. Kaggle **P100** (Internet ON) → run all → download CSV (in /kaggle/working)
3. Colab **CPU** runtime (or Codespaces) → run all → download CSV
Merge the three CSVs; that's the paper's hardware table. If T4 and P100 both show positive GPU savings at f≈0.4 while CPU is ~breakeven, the "deployment-dependent savings" claim rests on **two GPU architectures + CPU, all on free public compute** — which doubles as the paper's reproducibility statement.